In [ ]:
# IMPORTS Y CONFIGURACIÓN
import os
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np

load_dotenv()

# CONFIGURACIÓN CLIENTE
client = OpenAI(
    base_url="GITHUB_BASE_URL",
    api_key="GITHUB_TOKEN"
)

# RAG
documentos = [
    "El permiso de circulación se paga una vez al año en marzo y requiere SOAP y revisión técnica.",
    "Las patentes comerciales deben renovarse semestralmente en enero y julio.",
    "Las multas de tránsito pueden pagarse online o presencialmente en la municipalidad.",
    "La Municipalidad de Llanquihue ofrece servicios sociales, subsidios y atención comunitaria.",
]

# GENERACION DE EMBEDDINGS
def obtener_embedding(texto):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texto
    )
    return response.data[0].embedding

#PREPROCESAMIENTO DOCUMENTOS
doc_embeddings = [obtener_embedding(doc) for doc in documentos]


#COMPARACION TEXTOS
def similitud(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# RETRIEVAL
def buscar_contexto(pregunta, top_k=2):
    emb_pregunta = obtener_embedding(pregunta)

    similitudes = [
        (doc, similitud(emb_pregunta, emb_doc))
        for doc, emb_doc in zip(documentos, doc_embeddings)
    ]

    # ordenar por similitud
    similitudes.sort(key=lambda x: x[1], reverse=True)

    mejores = [doc for doc, _ in similitudes[:top_k]]

    return "\n".join(mejores)
    

# FUNCIÓN GENERAL IA 
def consultar_muni(pregunta, contexto, tipo="zero-shot"):
    try:
        #RAG: obtener contexto relevante
        contexto_rag = buscar_contexto(pregunta)

        contexto_final = f"""
        {contexto}

        Información relevante de la municipalidad:
        {contexto_rag}
        """

        # ZERO-SHOT
        if tipo == "zero-shot":
            messages = [
                {
                    "role": "system",
                    "content": contexto_final + 
                    ". Responde de forma clara, breve y útil para ciudadanos de Chile."
                },
                {"role": "user", "content": pregunta}
            ]

        # FEW-SHOT
        elif tipo == "few-shot":
            messages = [
                {
                    "role": "system",
                    "content": contexto_final + 
                    ". Responde como asistente municipal claro y práctico."
                },
                {"role": "user", "content": pregunta}
            ]

        # CHAIN OF THOUGHT
        elif tipo == "chain-of-thought":
            messages = [
                {
                    "role": "system",
                    "content": contexto_final + 
                    ". Responde siguiendo este formato:\n"
                    "1. Explicación breve\n"
                    "2. Pasos a seguir\n"
                    "3. Recomendación final"
                },
                {"role": "user", "content": pregunta}
            ]

        else:
            messages = [
                {"role": "system", "content": contexto_final},
                {"role": "user", "content": pregunta}
            ]

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.7,
            max_tokens=325
        )

        print("\nRespuesta:")
        print(response.choices[0].message.content, "\n")

    except Exception as e:
        print("Error:", e)

#ELEGIR TIPO DE RESPUESTA
def elegir_tipo_respuesta():
    print("\nTipo de respuesta:")
    print("1. Zero-shot")
    print("2. Few-shot")
    print("3. Chain of Thought")

    tipo_op = input("Elige tipo: ")

    tipo_map = {
        "1": "zero-shot",
        "2": "few-shot",
        "3": "chain-of-thought"
    }

    return tipo_map.get(tipo_op, "zero-shot")


# FUNCIONES MUNICIPALES
def permiso_circulacion():
    print("\nPERMISO DE CIRCULACIÓN")
    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()
    contexto = "Eres un asistente de la Municipalidad de Llanquihue experto en permisos de circulación en Chile"
    consultar_muni(pregunta, contexto, tipo)


def patentes_comerciales():
    print("\nPATENTES COMERCIALES")
    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()
    contexto = "Eres un asistente experto en patentes comerciales municipales en Chile"
    consultar_muni(pregunta, contexto)


def pago_multas():
    print("\nPAGO DE MULTAS")
    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()
    contexto = "Eres un asistente que ayuda con el pago de multas de tránsito en municipalidades chilenas"
    consultar_muni(pregunta, contexto)


def servicios_varios():
    print("\nSERVICIOS VARIOS")
    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()
    contexto = "Eres un asistente municipal que informa sobre servicios generales de la Municipalidad de Llanquihue"
    consultar_muni(pregunta, contexto)


# CHAT GENERAL
def chat_general():
    print("\nChat general (escribe 'salir')\n")

    tipo = elegir_tipo_respuesta()

    historial = [
        {"role": "system", "content": "Eres un asistente de la Municipalidad de Llanquihue."}
    ]

    while True:
        user_input = input("Tú: ")

        if user_input.lower() == "salir":
            break

        historial.append({"role": "user", "content": user_input})

        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=historial,
                temperature=0.7,
                max_tokens=225
            )

            respuesta = response.choices[0].message.content
            print("IA:", respuesta, "\n")

            historial.append({"role": "assistant", "content": respuesta})

        except Exception as e:
            print("Error:", e)


# MENÚ MUNICIPAL
def menu():
    while True:
        print("\n====== MUNICIPALIDAD DE LLANQUIHUE ======")
        print("1. Permiso de Circulación")
        print("2. Patentes Comerciales")
        print("3. Pago de Multas")
        print("4. Servicios Varios")
        print("5. Chat General")
        print("6. Salir")

        opcion = input("Selecciona una opción: ")

        if opcion == "1":
            permiso_circulacion()
        elif opcion == "2":
            patentes_comerciales()
        elif opcion == "3":
            pago_multas()
        elif opcion == "4":
            servicios_varios()
        elif opcion == "5":
            chat_general()
        elif opcion == "6":
            print("Saliendo...")
            break
        else:
            print("Opción inválida")


# EJECUCIÓN
if __name__ == "__main__":
    menu()


====== MUNICIPALIDAD DE LLANQUIHUE ======
1. Permiso de Circulación
2. Patentes Comerciales
3. Pago de Multas
4. Servicios Varios
5. Chat General
6. Salir


Selecciona una opción:  1



PERMISO DE CIRCULACIÓN


¿Qué deseas saber?:  dime lo que necesito saber sobre un permiso de circulacion



Tipo de respuesta:
1. Zero-shot
2. Few-shot
3. Chain of Thought


Elige tipo:  1



Respuesta:
El permiso de circulación es un documento que permite a tu vehículo transitar legalmente por las calles de Chile. Aquí tienes lo esencial:

1. **Pago Anual**: Debes pagarlo en marzo de cada año. Si lo haces en dos cuotas, la segunda vence en agosto.
   
2. **Requisitos**:
   - **SOAP** (Seguro Obligatorio de Accidentes Personales) vigente.
   - **Revisión Técnica** al día o Certificado de Homologación.
   - **Permiso de Circulación del año anterior**, si corresponde.

3. **Dónde Pagar**:
   - Presencial: En la Municipalidad de Llanquihue o en puntos habilitados.
   - Online: A través del sitio web oficial de la Municipalidad, si está disponible.

4. **Multas**:
   Si no pagas el permiso dentro del plazo, corres el riesgo de multas, intereses, y la imposibilidad de circular legalmente.

¿Necesitas ayuda adicional? 


====== MUNICIPALIDAD DE LLANQUIHUE ======
1. Permiso de Circulación
2. Patentes Comerciales
3. Pago de Multas
4. Servicios Varios
5. Chat General
6. Salir


Selecciona una opción:  6


Saliendo...
